# Visual ML Foundations: Molecular Fingerprints & Chemical Similarity
### Circular Fingerprints (ECFP/Morgan), Bit Vectors, and Tanimoto Coefficients

---

## 1. Concept Overview
In chemistry and drug discovery (such as the **Enveda CASMI** challenge), molecules cannot be fed directly into standard ML models as 2D sketches.
We convert molecules into **Extended Connectivity Fingerprints (ECFP)**:
1. Each atom and its circular neighborhood of radius $r$ is hashed into an integer.
2. The integers are folded into a fixed-length bit vector (e.g. 1024 or 2048 bits).
3. Chemical similarity between two molecules $A$ and $B$ is measured via the **Tanimoto Coefficient**:
$$T(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{N_c}{N_a + N_b - N_c}$$
Where $N_c$ is the number of shared bits, and $N_a, N_b$ are the bits set in $A$ and $B$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate 64-bit structural fingerprints for 4 chemical compounds
np.random.seed(42)
def generate_fp(base_bits, noise_rate=0.1, n_bits=64):
    fp = np.zeros(n_bits, dtype=int)
    fp[base_bits] = 1
    # Add random mutation bits
    flips = np.random.choice(n_bits, size=int(n_bits * noise_rate), replace=False)
    fp[flips] = 1 - fp[flips]
    return fp

# Base scaffold bits
aspirin_scaffold = [4, 12, 19, 27, 33, 45, 52]
mol_A = generate_fp(aspirin_scaffold, noise_rate=0.05)
mol_B = generate_fp(aspirin_scaffold, noise_rate=0.08) # Close analog
mol_C = generate_fp([2, 9, 21, 38, 59], noise_rate=0.1) # Unrelated alkaloid

def tanimoto_similarity(fp1, fp2):
    intersection = np.sum((fp1 == 1) & (fp2 == 1))
    union = np.sum((fp1 == 1) | (fp2 == 1))
    return intersection / union if union > 0 else 0.0

sim_AB = tanimoto_similarity(mol_A, mol_B)
sim_AC = tanimoto_similarity(mol_A, mol_C)

# Visualizing the bit vectors
fig, ax = plt.subplots(figsize=(14, 3.5))
fps_matrix = np.vstack([mol_A, mol_B, mol_C])
cax = ax.imshow(fps_matrix, cmap='Blues', aspect='auto', interpolation='nearest')
ax.set_yticks([0, 1, 2])
ax.set_yticklabels([
    f"Compound A (Reference)",
    f"Compound B (Analog, Sim={sim_AB:.2f})",
    f"Compound C (Unrelated, Sim={sim_AC:.2f})"
], fontsize=11, fontweight='bold')
ax.set_xlabel("Fingerprint Bit Position (0 - 63)")
ax.set_title("Molecular Bit Vector Alignment & Tanimoto Overlap", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Calculated Tanimoto Overlap: A-B = {sim_AB:.3f}, A-C = {sim_AC:.3f}")
